# Endpoint-regularization, multi-resolution bar and Slurm study

## Mandatory contract for users and AI agents

**This notebook follows `scripts/templates/boilerplate_run.ipynb`, the source of truth for run notebooks.** Keep its cell order, headings, guards, and adapter calls. Do not add, remove, reorder, merge, or redesign sections without the user's explicit permission in the current conversation.

At each of N=100, 200, 300, 400, and 500, and caps 1280, 320, and 40, this pipeline keeps the matched low-low and high-high smoothness/sharpness endpoints as independent maxima. The bar performs 1,000 broad random Fourier starts for at most 20,000 Adam steps, 2,000 monotone polish steps on every saved run, and loose L-BFGS-B refinement of all 1,000 starts in memory-safe batches. Each loose batch has its own elapsed-time limit, so the full population is attempted rather than being cut off by one global deadline. A detached monitor submits strict peak refinement to Slurm for each N/cap/endpoint and replaces an incumbent strict job whenever a saved exploratory or loose result has a higher J mol. It halts only after the bar pipeline ends and current strict refinement has completed for all 30 independent maxima. Every handoff explicitly resets optimizer state while preserving the selected controls.

In [ ]:
from ofc.notebook_workflow import RunNotebook
from ofc.seed_sensitivity import fixed_endpoint_seed_sensitivity_documents

run_name = "endpoint_regularization_u320_seed_sensitivity_gpu"
workflow = RunNotebook(run_name)

## Create the immutable configs

Edit the exposed arguments, then activate once. `resume_optimizer=False` deliberately restores controls with a fresh optimizer state at every stage boundary. The three ordered bar configs for each resolution/cap/endpoint are generated by reusable package code; strict Slurm configs are immutable challenger-specific records created by the monitor.

In [ ]:
Activated = False

reuse_existing = False
database = "results/bar_endpoint_seed1000_loose_u320.sqlite3"
strict_database = "results/slurm_endpoint_strict_u320.sqlite3"
resolutions = [100, 200, 300, 400, 500]
cap_endpoint_settings = {
    1280: {
        "low": {"adam_learning_rate": 0.15, "adam_beta1": 0.95, "adam_beta2": 0.999, "smoothness": 7.905694150420948e-9, "sharpness": 7.905694150420948e-10},
        "high": {"adam_learning_rate": 0.15, "adam_beta1": 0.95, "adam_beta2": 0.999, "smoothness": 7.905694150420947e-6, "sharpness": 7.905694150420948e-7},
    },
    320: {
        "low": {"adam_learning_rate": 0.05, "adam_beta1": 0.95, "adam_beta2": 0.99, "smoothness": 3.952847075210474e-9, "sharpness": 3.952847075210474e-10},
        "high": {"adam_learning_rate": 0.05, "adam_beta1": 0.95, "adam_beta2": 0.99, "smoothness": 3.9528470752104736e-6, "sharpness": 3.952847075210474e-7},
    },
    40: {
        "low": {"adam_learning_rate": 0.15, "adam_beta1": 0.9, "adam_beta2": 0.99, "smoothness": 3.952847075210474e-9, "sharpness": 1.5811388300841896e-9},
        "high": {"adam_learning_rate": 0.15, "adam_beta1": 0.9, "adam_beta2": 0.99, "smoothness": 3.9528470752104736e-6, "sharpness": 1.5811388300841896e-6},
    },
}
exploration_initialisations = 1_000
exploration_schedule = [(5_000, 1.5), (15_000, 0.75)]
all_polish_steps = 2_000
top_count = 1_000
batch_sizes = {100: 75, 200: 40, 300: 28, 400: 20, 500: 16}
loose_batch_sizes = {100: 40, 200: 25, 300: 16, 400: 12, 500: 8}
loose_max_elapsed_seconds = 2 * 60 * 60
strict_max_elapsed_seconds = 4 * 60 * 60
fourier_num_modes = 6
fourier_rms_amplitude = 0.8
fourier_intensity_fraction = 0.5
resume_optimizer = False

pipeline_documents = fixed_endpoint_seed_sensitivity_documents(
    cap_endpoint_settings=cap_endpoint_settings, resolutions=resolutions, database=database,
    exploration_initialisations=exploration_initialisations, exploration_schedule=exploration_schedule,
    all_polish_steps=all_polish_steps, batch_sizes=batch_sizes, loose_batch_sizes=loose_batch_sizes,
    top_count=top_count,
    loose_max_elapsed_seconds=loose_max_elapsed_seconds, strict_max_elapsed_seconds=strict_max_elapsed_seconds,
    fourier_num_modes=fourier_num_modes, fourier_rms_amplitude=fourier_rms_amplitude,
    fourier_intensity_fraction=fourier_intensity_fraction, resume_optimizer=resume_optimizer,
    include_strict=False, parameter_label_suffix="_bar_v3_loose1000",
)
pipeline_workflows = workflow.create_config_group(
    activated=Activated, documents=pipeline_documents, reuse_existing=reuse_existing,
)

## Run directly on `bar`'s GPU (detached)

This verifies JAX CUDA visibility and launches all 90 bar configs in dependency order inside one detached process. The second public adapter call starts the detached Slurm strict-incumbent monitor. `--continue-on-error` records a failed bar stage and proceeds; the monitor maintains separate maxima for both endpoints and halts after strict refinement completes at every N.

In [ ]:
Activated = False

selected_workflows = pipeline_workflows
queue_id = None
python_executable = "/home/gyqyan/charlie/.conda/envs/optical-feshbach-control/bin/python"
extra_arguments = ["--continue-on-error"]
detached = True
log_path = "logs/endpoint_regularization_u320_seed1000_loose_gpu.log"

active_queue_id = selected_workflows[0].run_on_bar_gpu_group(
    activated=Activated, additional_workflows=selected_workflows[1:],
    queue_id=queue_id, python_executable=python_executable, extra_arguments=extra_arguments,
    detached=detached, log_path=log_path,
)

strict_monitor_process_id = workflow.launch_strict_refinement_monitor(
    activated=Activated, bar_process_id=selected_workflows[0].local_process_id,
    endpoint_settings=cap_endpoint_settings, resolutions=resolutions,
    bar_database=database, strict_database=strict_database,
    state_path="logs/endpoint_strict_u320_state.json", python_executable=python_executable,
    exploration_initialisations=exploration_initialisations,
    loose_count=top_count, parameter_label_suffix="_bar_v3_loose1000",
    strict_max_elapsed_seconds=strict_max_elapsed_seconds, poll_seconds=60, objective_epsilon=1e-9,
    partition="zen5,epyc", slurm_time="04:15:00", cpus=2, memory="4G",
    agreement_objective_tolerance=0.01, agreement_control_tolerance=0.01,
    log_path="logs/endpoint_strict_u320_seed1000_monitor.log",
)

## Submit through Slurm (alternative)

The requested study is an ordered bar-GPU pipeline. Leave this alternative inactive; it exposes the standard boilerplate resources only for an explicitly redesigned submission.

In [ ]:
Activated = False

partition = "zen5,epyc"
time = "4-03:00:00"
cpus = 4
memory = "16G"
array = False
array_max_concurrent = None
job_name = None
extra_arguments = []

active_queue_id = selected_workflows[0].submit_slurm(
    activated=Activated, partition=partition, time=time, cpus=cpus, memory=memory,
    array=array, array_max_concurrent=array_max_concurrent, job_name=job_name,
    extra_arguments=extra_arguments,
)

## Query persisted data

Choose one bar stage offset: exploration=0, all-run polish=1, or all-1,000 loose=2. The default shows both u=1280 endpoints across resolution, allowing J mol and controls to be compared independently of the regularization penalty. Strict results live in the isolated Slurm database.

In [ ]:
stage_offset = 0  # 0=exploration, 1=all polish, 2=all-1,000 loose.
query_workflows = pipeline_workflows[stage_offset::3]
queue_id = None
config_run_rank = 1
statuses = None
filters = {"u_max": 1280}
sweep_parameters = ["N", "smoothness", "sharpness"]
require_saved_stage = True
allow_missing = True
limit = None
order_by = "run_id"
descending = False

query_result = query_workflows[0].query_group(
    additional_workflows=query_workflows[1:], queue_id=queue_id,
    config_run_rank=config_run_rank, statuses=statuses, filters=filters,
    sweep_parameters=sweep_parameters, require_saved_stage=require_saved_stage,
    allow_missing=allow_missing, limit=limit, order_by=order_by, descending=descending,
)

## Figure display and saving

In [ ]:
save_figure = None
figure_format = "png"
preview_dpi = 240
save_dpi = 600

## Unified sweep summary

The history median/spread and objective strip use only the runs shown in each resolution row. Dashed vertical lines mark the two exploratory learning-rate stages.

In [ ]:
sweep_parameter = "smoothness"
history_points = 1200
summary_figure = query_result.plot_summary(sweep_parameter=sweep_parameter, history_points=history_points)
workflow.present_figure(summary_figure, "01_sweep_summary", save_figure=save_figure, figure_format=figure_format, preview_dpi=preview_dpi, save_dpi=save_dpi)

## Double sweep summary

In [ ]:
separate_sweep_parameter = "N"
colour_sweep_parameter = "sharpness"
history_points = 1200
double_sweep_figure = query_result.plot_double_sweep_summary(separate_sweep_parameter=separate_sweep_parameter, colour_sweep_parameter=colour_sweep_parameter, history_points=history_points)
workflow.present_figure(double_sweep_figure, "02_double_sweep_summary", save_figure=save_figure, figure_format=figure_format, preview_dpi=preview_dpi, save_dpi=save_dpi)

## Triple sweep summary

In [ ]:
row_sweep_parameter = "N"
column_sweep_parameter = "u_max"
colour_sweep_parameter = "smoothness"
history_points = 1200
triple_sweep_figure = query_result.plot_triple_sweep_summary(row_sweep_parameter=row_sweep_parameter, column_sweep_parameter=column_sweep_parameter, colour_sweep_parameter=colour_sweep_parameter, history_points=history_points)
workflow.present_figure(triple_sweep_figure, "03_triple_sweep_summary", save_figure=save_figure, figure_format=figure_format, preview_dpi=preview_dpi, save_dpi=save_dpi)